In [ ]:
# CHANGE: Installed groq instead of google-generativeai
!pip install -q -U groq pillow tqdm

In [2]:
from groq import Groq
import json
import os
import pathlib
import time
import base64
from tqdm.auto import tqdm
from PIL import Image, ImageFile
import io

In [3]:
from google.colab import drive
drive.mount('/gdrive')

Mounted at /gdrive


In [4]:
from google.colab import userdata
ImageFile.LOAD_TRUNCATED_IMAGES = True

In [ ]:
api_keys = [
    "",
    "",
]
current_key_index = 0
client = Groq(api_key=api_keys[current_key_index])
print(f"✅ Loaded {len(api_keys)} API keys. Starting with key #1.")

In [ ]:
# ============================================================
# CELL 2.5: Validate All API Keys
# ============================================================
valid_keys = []

for i, key in enumerate(api_keys):
    try:
        test_client = Groq(api_key=key)
        test_client.chat.completions.create(
            model="meta-llama/llama-4-scout-17b-16e-instruct",
            messages=[{"role": "user", "content": "Say OK"}],
            max_tokens=2,
        )
        print(f"✅ Key #{i+1}: Valid")
        valid_keys.append(key)
    except Exception as e:
        print(f"❌ Key #{i+1}: INVALID — {str(e)[:60]}")
    time.sleep(1)

# Replace the list with only working keys
api_keys = valid_keys
current_key_index = 0
client = Groq(api_key=api_keys[current_key_index])
print(f"\n🎯 {len(api_keys)} valid keys out of {len(api_keys) + (len(api_keys) != len(valid_keys))} total. Ready to go.")


In [22]:
def preprocess_for_vlm(image):
    if image.mode != 'RGB':
        image = image.convert('RGB')
    byte_stream = io.BytesIO()
    image.save(byte_stream, format='JPEG', quality=95)
    return base64.b64encode(byte_stream.getvalue()).decode('utf-8')

prompt = """Extract every food or drink item and its price from this menu image.

Rules:
- Keep item names and prices exactly as written in the image.
- If an item has multiple sizes, list each size as a separate entry like "بيتزا مارجريتا (صغير)" and "بيتزا مارجريتا (كبير)".
- Only include items that have a visible price. Skip anything without a price.

Return a JSON object with one key "items" containing an array. Each item has "name" (string) and "price" (string)."""

image_folder = pathlib.Path("/gdrive/MyDrive/data")
output_folder = pathlib.Path("/gdrive/MyDrive/menu_json_ground_truth")
output_folder.mkdir(parents=True, exist_ok=True)

valid_extensions = {'.jpg', '.jpeg', '.png', '.webp'}
image_paths = [
    path for path in image_folder.iterdir()
    if path.suffix.lower() in valid_extensions
]

print(f"Found {len(image_paths)} images to process.")


Found 3420 images to process.


In [ ]:
# ============================================================
# CELL 4: Processing Loop (Auto Key Rotation + Garbage Cleanup)
# ============================================================
MIN_ITEMS = 3  # Delete images with fewer items than this
deleted_garbage = 0

for img_path in tqdm(image_paths, desc="Processing Menus"):
    json_path = output_folder / f"{img_path.stem}.json"
    if json_path.exists():
        continue

    # Skip if image was already deleted by a previous run
    if not img_path.exists():
        continue

    success = False
    while not success:
        try:
            raw_img = Image.open(img_path)
            print(f"\r🔑 {api_keys[current_key_index]} | {img_path.name}", end="")
            base64_image = preprocess_for_vlm(raw_img)

            response = client.chat.completions.create(
                model="meta-llama/llama-4-scout-17b-16e-instruct",
                messages=[
                    {
                        "role": "user",
                        "content": [
                            {"type": "text", "text": prompt},
                            {
                                "type": "image_url",
                                "image_url": {
                                    "url": f"data:image/jpeg;base64,{base64_image}",
                                },
                            },
                        ],
                    }
                ],
                temperature=0.0,
                response_format={"type": "json_object"}
            )

            parsed_json = json.loads(response.choices[0].message.content)

            # === GARBAGE CHECK ===
            items = parsed_json.get("items", [])
            if len(items) < MIN_ITEMS:
                print(f"\n🗑️ {img_path.name}: only {len(items)} items. Deleting image.")
                os.remove(img_path)
                deleted_garbage += 1
                success = True
            else:
                with open(json_path, "w", encoding="utf-8") as f:
                    json.dump(parsed_json, f, ensure_ascii=False, indent=2)
                success = True

            time.sleep(4.1)

        except Exception as e:
            error_str = str(e)

            if "429" in error_str or "rate_limit" in error_str.lower() or "quota" in error_str.lower() or "limit" in error_str.lower() or "organization_restricted" in error_str.lower() or "invalid_api_key" in error_str.lower():
                current_key_index = (current_key_index + 1) % len(api_keys)

                if current_key_index == 0:
                    print(f"\n⏳ All {len(api_keys)} keys cycled. Waiting 60s for quota reset...")
                    time.sleep(60)

                print(f"\n🔄 Switching to key #{current_key_index + 1}/{len(api_keys)}...")
                client = Groq(api_key=api_keys[current_key_index])
                time.sleep(2)
            else:
                print(f"\n❌ Skipping {img_path.name}: {e}")
                success = True  # skip this image, move on

print(f"\n✅ Extraction complete!")
print(f"🔑 Used key #{current_key_index + 1} as the last active key.")
print(f"🗑️ Deleted {deleted_garbage} garbage images (fewer than {MIN_ITEMS} items).")


In [69]:
# ============================================================
# CELL 5: Remove Images With No JSON (Corrupted/Failed)
# ============================================================
image_paths_check = [
    path for path in image_folder.iterdir()
    if path.suffix.lower() in valid_extensions
]

deleted_count = 0
for img_path in image_paths_check:
    json_path = output_folder / f"{img_path.stem}.json"
    if not json_path.exists():
        os.remove(img_path)
        deleted_count += 1

print(f"✅ Deleted {deleted_count} images with no matching JSON.")


✅ Deleted 0 images with no matching JSON.


In [70]:
# ============================================================
# CELL 6: Remove Duplicate Image Names (Different Extensions)
# ============================================================
image_paths_dedup = [
    path for path in image_folder.iterdir()
    if path.suffix.lower() in valid_extensions
]

seen_stems = {}
duplicates_deleted = 0

for img_path in image_paths_dedup:
    stem = img_path.stem
    if stem in seen_stems:
        print(f"🗑️ Deleting duplicate: {img_path.name} (conflicts with {seen_stems[stem].name})")
        os.remove(img_path)
        duplicates_deleted += 1
    else:
        seen_stems[stem] = img_path

print(f"✅ Removed {duplicates_deleted} duplicate(s).")


✅ Removed 0 duplicate(s).


In [71]:
# ============================================================
# CELL 7: Final Count & Verification
# ============================================================
image_count = len([f for f in image_folder.iterdir() if f.suffix.lower() in valid_extensions])
json_count = len([f for f in output_folder.iterdir() if f.suffix.lower() == '.json'])

print(f"🖼️ Images: {image_count}")
print(f"📄 JSONs:  {json_count}")

if image_count == json_count:
    print("\n✅ PERFECT MATCH! Dataset is ready.")
else:
    print(f"\n⚠️ MISMATCH: difference of {abs(image_count - json_count)} files.")


🖼️ Images: 3080
📄 JSONs:  3080

✅ PERFECT MATCH! Dataset is ready.
